# Imports

In [ ]:
import sys
import pandas as pd
import os
import sys
import json
import re
from IPython.display import HTML

# 1. Path Setup for Docker/Local consistency
current_dir = os.getcwd()
if os.path.basename(current_dir) == 'ETL':
    project_root = os.path.abspath(os.path.join(current_dir, '..'))
else:
    project_root = current_dir

if project_root not in sys.path:
    sys.path.append(project_root)

# 2. Ensure working directory is correct for data loading
if os.path.basename(os.getcwd()) != 'ETL' and os.path.exists('ETL'):
    os.chdir('ETL')

print(f"Working Directory: {os.getcwd()}")

# Configurations

In [ ]:
# Define relative paths
INPUT_FILE_EVENTS = "../data_temp/events_1_intermediate.json"
OUTPUT_FOLDER = "../data_temp/"
OUTPUT_FILE = os.path.join(OUTPUT_FOLDER, "events_2_speakers_intermediate.json")

# Simple check to verify the file is where we think it is
if os.path.exists(INPUT_FILE_EVENTS):
    print(f"✅ Setup complete. Input file found: {INPUT_FILE_EVENTS}")
else:
    print(f"❌ Warning: Input file NOT found at {INPUT_FILE_EVENTS}")

# Ensure the output directory exists
if os.path.exists(OUTPUT_FOLDER):
    print(f"✅ Setup complete. Output folder found: {OUTPUT_FOLDER}")
else:
    os.makedirs(OUTPUT_FOLDER)
    print(f"✅ Created Output folder: {OUTPUT_FOLDER}")

# Load the raw data

In [ ]:
try:
    events_df = pd.read_json(INPUT_FILE_EVENTS)
    print(f"✅ Successfully loaded {len(events_df)} records.")
    
    display(events_df.head(3)) 
    
    print("\n📑 Available columns:", *events_df.columns, sep="\n")
    
except FileNotFoundError:
    print(f"Error: The file {INPUT_FILE_EVENTS} was not found.")

# Add speaker column

In [ ]:
# Add the speaker column with an empty string as the default value
events_df['speaker'] = ""

# Define the final column order
final_columns = ['id', 'title', 'date', 'speaker', 'location', 'category', 'eventGuests', 'descriptionText', 'url']

# Reorder the DataFrame based on final_columns
# We use .copy() to avoid SettingWithCopy warnings if you perform further operations
events_df = events_df[final_columns].copy()

print("✅ Added 'speaker' column and reordered columns.")
print(f"New shape: {events_df.shape}")

display(events_df.head(3))

**[INFO]**  
Wix Events does not contain a dedicated field for speakers. The speaker is usually mentioned inside the description text for the event.  
All events are available from the front end, and randomly some was opened to search for some kind of pattern which could be used when selecting a speaker for a given event.  
The following patterns were observed:  
- "ki vezeti"  
- "ki segít neked"  
- "Czimer Györgyi"   
- "Tótiván Tibor"  

These will be discussed as` Filter_Case_1` or _2 or _3 or _4.  

#### Reusable_Function_1

In [ ]:
def extract_snippet(text, keyword, length=500):
    """
    Finds a keyword in text (case-insensitive), returns a snippet 
    of the keyword plus the following N characters, with the keyword highlighted.
    """
    if not isinstance(text, str):
        return None

    # 1. Find the first occurrence to establish the starting point
    pattern = re.compile(re.escape(keyword), re.IGNORECASE)
    match = pattern.search(text)
    
    if match:
        start_index = match.start()
        # Extract the snippet (keyword + N characters following)
        raw_snippet = text[start_index : start_index + len(keyword) + length]
        
        # 2. Highlight ALL occurrences of the keyword within this specific snippet
        # We use re.escape to handle special characters and re.IGNORECASE for case-insensitivity
        highlighted = re.sub(
            f'({re.escape(keyword)})', 
            r'<mark style="background-color: lightyellow;">\1</mark>', 
            raw_snippet, 
            flags=re.IGNORECASE
        )
    
        return highlighted
    
    return None

# Filter_Case_1

In [ ]:
filter_term_1 = "ki vezeti"

In [ ]:
filtered_df_1 = events_df[
    events_df['descriptionText'].str.contains(filter_term_1, case=False, na=False)
].copy()

# Apply the Reusable_Function_1
filtered_df_1['snippet'] = filtered_df_1['descriptionText'].apply(
    lambda x: extract_snippet(x, filter_term_1)
)

results_filter_term_1 = filtered_df_1[filtered_df_1['snippet'].notna()][['id', 'title', 'snippet']]


print(f"✅ Found {len(results_filter_term_1)} matches.")

# To actually see the highlighted HTML in a Jupyter notebook, you must use display(HTML(...))
display(HTML(results_filter_term_1.to_html(escape=False)))

In [ ]:
speaker_filtered_df_1 = "Dr. Prezenszki Zsuzsanna"

In [ ]:
# Update the 'speaker' column using the index of your filtered results
events_df.loc[filtered_df_1.index, 'speaker'] = speaker_filtered_df_1

print(f"✅ Updated {len(filtered_df_1)} rows with the new speaker.")

display(events_df.loc[filtered_df_1.index, ['id', 'title', 'speaker']].head(4))

In [ ]:
# Count events with and without a speaker
total_has_speaker = (events_df['speaker'] != "").sum()
total_no_speaker = (events_df['speaker'] == "").sum()

print(f"📊 Global Speaker Statistics (Total Dataset):")
print(f"    - Events with a speaker: {total_has_speaker}")
print(f"    - Events without a speaker: {total_no_speaker}")
print(f"    - Total events: {len(events_df)}")

# Filter_Case_2

In [ ]:
filter_term_2 = "ki segít neked"

In [ ]:
# Filter: Text contains keyword AND speaker is empty ("")
# Added .str.strip() to ensure spaces don't bypass the "" check
mask_filter_term_2 = (events_df['descriptionText'].str.contains(filter_term_2, case=False, na=False)) & \
       (events_df['speaker'].str.strip() == "")

filtered_df_2 = events_df[mask_filter_term_2].copy()

# Apply the Reusable_Function_1 
filtered_df_2['snippet'] = filtered_df_2['descriptionText'].apply(
    lambda x: extract_snippet(x, filter_term_2)
)

results_filter_term_2 = filtered_df_2[filtered_df_2['snippet'].notna()][['id', 'title', 'snippet']]

print(f"✅ Found {len(results_filter_term_2)} matches with no speaker assigned.")

display(HTML(results_filter_term_2.to_html(escape=False)))

**[INFO]**  
Based on the output, three different speakers could be applied:  
- "Dr. Prezenszki Zsuzsanna"  
- "dr. Matuszka István"  
- "Győrfi Andrea"  

***Note***: When *"Győrfi Andrea"* is a speaker, *"Dr. Prezenszki Zsuzsanna"* is also mentioned as a person giving a recommendation.   

#### Reusable_Function_2

In [ ]:
# Create a new highlighted version for the sub-term specifically
def highlight_subterm(text, sub_term):
    if not isinstance(text, str): return text
    # Highlight subterm in light blue
    return re.sub(
        f'({re.escape(sub_term)})', 
        r'<mark style="background-color: lightblue;">\1</mark>', 
        text, 
        flags=re.IGNORECASE
    )

### Sub-filter terms as an extensions for `filter_term_2` 

There are three sub_filter terms:
- "Matuszka István"  
- "Győrfi Andrea"  
- "Dr. Prezenszki Zsuzsanna"  

In [ ]:
sub_filter_term_2_1 = "Matuszka István"

In [ ]:
# We use case=False to be safe (catches "Matuszka", "MATUSZKA", etc.)
filtered_df_2_sub_term_1 = filtered_df_2[
    filtered_df_2['descriptionText'].str.contains(sub_filter_term_2_1, case=False, na=False)
].copy()

# Apply the Reusable_Function_2 to highlight the sub-term within the existing snippet
filtered_df_2_sub_term_1['snippet'] = filtered_df_2_sub_term_1['snippet'].apply(
    lambda x: highlight_subterm(x, sub_filter_term_2_1)
)

print(f"✅ Found {len(filtered_df_2_sub_term_1)} matches.")

display(HTML(filtered_df_2_sub_term_1[['id', 'title', 'snippet']].to_html(escape=False)))

In [ ]:
speaker_filtered_df_2_1 = "dr. Matuszka István"

In [ ]:
# Update the 'speaker' column using the index of your filtered results
events_df.loc[filtered_df_2_sub_term_1.index, 'speaker'] = speaker_filtered_df_2_1

print(f"✅ Updated {len(filtered_df_2_sub_term_1)} rows with the new speaker.")

display(events_df.loc[filtered_df_2_sub_term_1.index, ['id', 'title', 'speaker']])

In [ ]:
# Statistics specifically for the CURRENT working batch (filtered_df_2)
# This helps to see how much of 'ki segít neked' is left to process
batch_filtered_2_has_speaker = (events_df.loc[filtered_df_2.index, 'speaker'] != "").sum()
batch_filtered_2_no_speaker = (events_df.loc[filtered_df_2.index, 'speaker'] == "").sum()

print(f"📊 Batch Statistics ('ki segít neked' group):")
print(f"   - Already assigned in this batch: {batch_filtered_2_has_speaker}")
print(f"   - Remaining to assign in this batch: {batch_filtered_2_no_speaker}")


In [ ]:
sub_filter_term_2_2 = "Győrfi Andrea"
helper_sub_filter_term_2_2 = "Simonton"

In [ ]:
# Filter: Text contains keyword AND speaker is empty ("")
# Added .str.strip() to ensure spaces don't bypass the "" check
mask_sub_filter_term_2_2 = (filtered_df_2['descriptionText'].str.contains(sub_filter_term_2_2, case=False, na=False)) & \
       (filtered_df_2['title'].str.contains(helper_sub_filter_term_2_2, case=False, na=False)) & \
       (events_df.loc[filtered_df_2.index,'speaker'].str.strip() == "")

filtered_df_2_sub_term_2 = filtered_df_2[mask_sub_filter_term_2_2].copy()

# Apply the ReusableFunction_2 to highlight the sub-term within the existing snippet
filtered_df_2_sub_term_2['snippet'] = filtered_df_2_sub_term_2['snippet'].apply(
    lambda x: highlight_subterm(x, sub_filter_term_2_2)
)

print(f"✅ Found {len(filtered_df_2_sub_term_2)} matches.")

display(HTML(filtered_df_2_sub_term_2[['id', 'title', 'speaker', 'snippet']].to_html(escape=False)))

[INFO]  
The previous cell displayed one case out of five where "Győrfi Andrea" was not the speaker but was detected in the `descriptionText` field.   
In the next cell, based on the event ID, that case was examined thoroughly.  

In [ ]:
filtered_df_2_sub_term_2.loc[41]

In [ ]:
# 1. Create a 100% isolated copy of just row 41
row_41_audit = filtered_df_2_sub_term_2.loc[[41]].copy()

# 2. Extract the raw text from the copy to process it
raw_text_41 = row_41_audit.loc[41, 'descriptionText']

# 3. Process the snippet into a temporary variable
# This does NOT save back to row_41_audit or the original DF.
target_name = "Andrea"

# Apply the Reusable_Function_1 and Reusable_Function_2 in sequence to get the highlighted snippet
preview_snippet = highlight_subterm(
    extract_snippet(raw_text_41, target_name, length=2500), 
    target_name
)

# 4. Display 
display(HTML("Audit Table (Copy):"))
display(HTML(row_41_audit.drop(columns=['descriptionText']).to_html(escape=False)))

display(HTML("Snippet Preview:"))
display(HTML(preview_snippet))

**[INFO]**  
The examined row was excluded from updating the `speaker` field with "Győrfi Andrea".

In [ ]:
speaker_filtered_df_2_2 = "Győrfi Andrea"

In [ ]:
# 1. Create a list of indices EXCLUDING 41
indices_to_update = filtered_df_2_sub_term_2.index.drop(41)

# 2. Update the 'speaker' column using those specific indices
# We use .loc[indices_to_update] to ensure 41 is skipped
events_df.loc[indices_to_update, 'speaker'] = speaker_filtered_df_2_2

# 3. Print the count (subtracting 1 since we skipped row 41)
print(f"✅ Updated {len(indices_to_update)} rows with the new speaker (Row 41 skipped).")

# 4. Display the results to verify
display(events_df.loc[filtered_df_2_sub_term_2.index, ['id', 'title', 'speaker']])

In [ ]:
# Statistics specifically for the CURRENT working batch (filtered_df_2)
# This helps to see how much of 'ki segít neked' is left to process
batch_filtered_2_has_speaker = (events_df.loc[filtered_df_2.index, 'speaker'] != "").sum()
batch_filtered_2_no_speaker = (events_df.loc[filtered_df_2.index, 'speaker'] == "").sum()

print(f"📊 Batch Statistics ('ki segít neked' group):")
print(f"   - Already assigned in this batch: {batch_filtered_2_has_speaker}")
print(f"   - Remaining to assign in this batch: {batch_filtered_2_no_speaker}")

**[NOTE]**  
The `filtered_df_2` is a static copy created earlier.  
To see only  the rows that are actually still empty, you need to pull the latest speaker values from the main `events_df`.

In [ ]:
# Create a fresh copy to work with
rest_filtered_df_2 = filtered_df_2.copy()

# Sync the speaker column from the master events_df to get the latest updates
rest_filtered_df_2['speaker'] = events_df.loc[rest_filtered_df_2.index,
'speaker']

# Now filter for rows that are still empty
rest_filtered_df_2 = rest_filtered_df_2[rest_filtered_df_2['speaker'] ==
""].copy()
   
print(f"✅ Found {len(rest_filtered_df_2)} total events matching {filter_term_2}' that still need a speaker.")

# Use HTML display to ensure the highlighted snippets are rendered correctly
display(HTML(rest_filtered_df_2[['id', 'title', 'speaker','snippet']].to_html(escape=False)))

In [ ]:
speaker_filtered_df_2_3 = "Dr. Prezenszki Zsuzsanna"

In [ ]:
# Update the 'speaker' column using the index of your filtered results
events_df.loc[rest_filtered_df_2.index, 'speaker'] = speaker_filtered_df_2_3

print(f"✅ Updated {len(rest_filtered_df_2)} rows with the new speaker.")

display(events_df.loc[rest_filtered_df_2.index, ['id', 'title', 'speaker']])

In [ ]:
# Statistics specifically for the CURRENT working batch (filtered_df_2)
# This helps to see how much of 'ki segít neked' is left to process
batch_filtered_2_has_speaker = (events_df.loc[filtered_df_2.index, 'speaker'] != "").sum()
batch_filtered_2_no_speaker = (events_df.loc[filtered_df_2.index, 'speaker'] == "").sum()

print(f"📊 Batch Statistics ('ki segít neked' group):")
print(f"   - Already assigned in this batch: {batch_filtered_2_has_speaker}")
print(f"   - Remaining to assign in this batch: {batch_filtered_2_no_speaker}")

In [ ]:
# Count events with and without a speaker
total_has_speaker = (events_df['speaker'] != "").sum()
total_no_speaker = (events_df['speaker'] == "").sum()

print(f"📊 Global Speaker Statistics (Total Dataset):")
print(f"    - Events with a speaker: {total_has_speaker}")
print(f"    - Events without a speaker: {total_no_speaker}")
print(f"    - Total events: {len(events_df)}")

# Filter_Case_3

In [ ]:
filter_term_3 = "Czimer Györgyi"

In [ ]:
# 1. Set Pandas to show the full content of columns
pd.set_option('display.max_colwidth', None)

filtered_df_3 = events_df[
    events_df['descriptionText'].str.contains(filter_term_3, case=False, na=False)
].copy()

# Apply the Reusable_Function_1
filtered_df_3['snippet'] = filtered_df_3['descriptionText'].apply(
    lambda x: extract_snippet(x, filter_term_3, length=1500)
)

# 2. Select BOTH 'snippet' and 'descriptionText'
results_filter_term_3 = filtered_df_3[filtered_df_3['snippet'].notna()][['id', 'title', 'snippet', 'url']]

print(f"✅ Found {len(results_filter_term_3)} matches.")

# 3. Display the HTML
display(HTML(results_filter_term_3.to_html(escape=False, index=False)))

In [ ]:
# 1. Set Pandas to show the full content of columns
pd.set_option('display.max_colwidth', None)

mask_filter_term_3 = (events_df['descriptionText'].str.contains(filter_term_3, case=False, na=False)) & (events_df['title'].str.contains("Meseterápia útkeresőknek", case=False, na=False)) 

filtered_df_3 = events_df[mask_filter_term_3].copy()

# Apply the Reusable Function-1
filtered_df_3['snippet'] = filtered_df_3['descriptionText'].apply(
    lambda x: extract_snippet(x, filter_term_3, length=1500)
)

# 2. Select BOTH 'snippet' and 'descriptionText'
results_filter_term_3 = filtered_df_3[filtered_df_3['snippet'].notna()][['id', 'title', 'snippet']]

print(f"✅ Found {len(results_filter_term_3)} matches.")

# 3. Display the HTML
display(HTML(results_filter_term_3.to_html(escape=False, index=False)))

In [ ]:
speaker_filtered_df_3 = "Czimer Györgyi"

In [ ]:
# Update the 'speaker' column using the index of your filtered results
events_df.loc[filtered_df_3.index, 'speaker'] = speaker_filtered_df_3

print(f"✅ Updated {len(filtered_df_3)} rows with the new speaker.")

display(events_df.loc[filtered_df_3.index, ['id', 'title', 'speaker']])

# Filter_Case_4

In [ ]:
filter_term_4 = "Tótiván Tibor"

In [ ]:
filtered_df_4 = events_df[events_df['descriptionText'].str.contains(filter_term_4, case=False, na=False)].copy()

filtered_df_4['snippet'] = filtered_df_4['descriptionText'].apply(
    lambda x: extract_snippet(x, filter_term_4, length=250)
)

print(f"✅ Found {len(filtered_df_4)} matches.")

display(HTML(filtered_df_4[['id', 'title', 'snippet']].to_html(escape=False, index=False)))

In [ ]:
speaker_filtered_df_4 = "Tótiván Tibor"

In [ ]:
# Update the 'speaker' column using the index of your filtered results
events_df.loc[filtered_df_4.index, 'speaker'] = speaker_filtered_df_4

print(f"✅ Updated {len(filtered_df_4)} rows with the new speaker.")

display(events_df.loc[filtered_df_4.index, ['id', 'title', 'speaker']])

In [ ]:
# Count events with and without a speaker
total_has_speaker = (events_df['speaker'] != "").sum()
total_no_speaker = (events_df['speaker'] == "").sum()

print(f"📊 Global Speaker Statistics (Total Dataset):")
print(f"    - Events with a speaker: {total_has_speaker}")
print(f"    - Events without a speaker: {total_no_speaker}")
print(f"    - Total events: {len(events_df)}")

# Export to JSON

In [ ]:
# force_ascii=False is crucial for keeping Hungarian characters like ő, ú, é
if pd.api.types.is_datetime64_any_dtype(events_df['date']):
    events_df['date'] = events_df['date'].dt.strftime('%Y-%m-%d')

# date_format='iso' ensures dates are readable strings
# categories will be included as long as they are in the events_df
events_df.to_json(
    OUTPUT_FILE, 
    orient='records', 
    indent=4, 
    force_ascii=False, 
    date_format='iso'
)

print(f"✅ Success! Cleaned data is saved to: {OUTPUT_FILE}")